# Module 06 — Convolutional Neural Networks
## 06-01 Convolution from Scratch

**Objective:** Implement 2-D cross-correlation from scratch using nested loops and the
efficient im2col strategy, visualise classic and learned kernels, and train a compact CNN
on MNIST to observe what feature maps capture at each layer.

**Prerequisites:** 05-02 (Multilayer Perceptrons), 05-08 (Weight Initialisation), 05-09 (Optimisers).


## Part 0 — Setup & Prerequisites

Convolution is the primitive that distinguishes CNNs from MLPs.  Instead of dense
connections ($784 \times 256 = 200{,}704$ parameters for one MLP layer), a convolutional
layer slides a small **kernel** (e.g., $3 \times 3 = 9$ parameters) across the spatial
dimensions — sharing weights at every position and preserving spatial structure.

In this notebook we:
1. Derive and implement single- and multi-channel 2-D cross-correlation from scratch.
2. Explore hand-crafted kernels (Sobel edge detectors, Gaussian blur, Laplacian).
3. Derive the output-dimension formula and show how padding / stride affect it.
4. Wrap the im2col approach into `Conv2dScratch` and verify against `nn.Conv2d`.
5. Train a compact CNN on MNIST and visualise feature maps before and after training.


In [ ]:
import sys
import math
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")

import random
import matplotlib.patches as mpatches
print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy  : {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA   : {torch.version.cuda}")
    print(f"GPU    : {torch.cuda.get_device_name(0)}")


In [ ]:

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# ── Hyperparameters & constants ───────────────────────────────────────────────
BATCH_SIZE    = 256
NUM_EPOCHS    = 5
LEARNING_RATE = 1e-3
NUM_CLASSES   = 10
DATA_ROOT     = "data/"

# MNIST normalisation statistics
MNIST_MEAN = (0.1307,)
MNIST_STD  = (0.3081,)

print(f"Batch size    : {BATCH_SIZE}")
print(f"Epochs        : {NUM_EPOCHS}")
print(f"Learning rate : {LEARNING_RATE}")
print(f"Device        : {device}")


### Dataset: MNIST

MNIST — 70,000 grey-scale $28 \times 28$ digit images (10 classes, 0–9) — is the ideal
sandbox for learning convolution because:
- The single input channel keeps the maths easy to follow.
- $28 \times 28$ is small enough to visualise every feature map at full resolution.
- Training finishes quickly even on CPU, so we can iterate fast.

We use the **official 60 K / 10 K** train / test split, reserving 10 % of training data
as a validation set.


In [ ]:
# ── Load MNIST ────────────────────────────────────────────────────────────────
transform_mnist = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

full_train = torchvision.datasets.MNIST(
    root=DATA_ROOT, train=True, download=True, transform=transform_mnist
)
test_set = torchvision.datasets.MNIST(
    root=DATA_ROOT, train=False, download=True, transform=transform_mnist
)

# 90/10 split of the official training set
generator  = torch.Generator().manual_seed(SEED)
train_size = int(0.9 * len(full_train))
val_size   = len(full_train) - train_size
train_set, val_set = random_split(full_train, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())

print(f"Train samples : {len(train_set):,}")
print(f"Val   samples : {len(val_set):,}")
print(f"Test  samples : {len(test_set):,}")
print(f"Image shape   : {full_train[0][0].shape}  (C x H x W)")

# ── Sample grid ───────────────────────────────────────────────────────────────
fig_data, axes_data = plt.subplots(3, 10, figsize=(15, 5))
class_shown: dict[int, bool] = {d: False for d in range(10)}
row0_count = 0
for img_t, lbl in full_train:
    if not class_shown[lbl]:
        axes_data[0][lbl].imshow(img_t.squeeze().numpy(), cmap="gray")
        axes_data[0][lbl].set_title(str(lbl), fontsize=10)
        axes_data[0][lbl].axis("off")
        class_shown[lbl] = True
        row0_count += 1
    if row0_count == 10:
        break

rng_vis = np.random.default_rng(SEED)
sample_idxs = rng_vis.choice(len(full_train), size=20, replace=False)
for plot_idx, data_idx in enumerate(sample_idxs):
    row = 1 + plot_idx // 10
    col = plot_idx % 10
    img_t, lbl = full_train[int(data_idx)]
    axes_data[row][col].imshow(img_t.squeeze().numpy(), cmap="gray")
    axes_data[row][col].set_title(str(lbl), fontsize=8)
    axes_data[row][col].axis("off")

plt.suptitle("MNIST Samples (row 0: one per class; rows 1-2: random)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Pixel statistics (raw, un-normalised images) ──────────────────────────────
raw_transform = transforms.ToTensor()
raw_mnist     = torchvision.datasets.MNIST(root=DATA_ROOT, train=False,
                                            download=False, transform=raw_transform)
pixel_vals    = np.array([raw_mnist[i][0].numpy().ravel() for i in range(500)]).ravel()

label_counts = Counter(int(full_train[i][1]) for i in range(len(full_train)))
print("\nClass distribution (training set):")
for digit in range(10):
    bar = "~" * (label_counts[digit] // 200)
    print(f"  Digit {digit}: {label_counts[digit]:,}  {bar}")

print(f"\nPixel stats (500 test images, raw [0,1]):")
print(f"  mean={pixel_vals.mean():.4f}  std={pixel_vals.std():.4f}  "
      f"min={pixel_vals.min():.4f}  max={pixel_vals.max():.4f}")

# Keep raw sample image for kernel visualisations
raw_sample_img = raw_mnist[0][0].squeeze().numpy()   # (28, 28)
raw_sample_lbl = raw_mnist[0][1]


## Part 1 — The Convolution Operation from Scratch

### 1.1 What is 2-D Cross-Correlation?

What deep-learning frameworks call "convolution" is technically **cross-correlation**:
a kernel $\mathbf{K} \in \mathbb{R}^{K_H \times K_W}$ slides over an input
$\mathbf{X} \in \mathbb{R}^{H \times W}$ and at each position $(h, w)$ computes:

$$
Z_{h,w} = \sum_{i=0}^{K_H-1}\sum_{j=0}^{K_W-1}
           X_{\,h \cdot s + i,\; w \cdot s + j} \cdot K_{i,j} + b
$$

where $s$ is the **stride**.  True convolution flips the kernel before sliding — the
distinction vanishes once weights are learned, so the field uses cross-correlation.

**Output spatial size** (per axis):
$$
\text{out\_size} = \left\lfloor \frac{\text{in\_size} + 2p - K}{s} \right\rfloor + 1
$$

where $p$ is zero-padding added on each side.


In [ ]:
# ── 1-A: Naive nested-loop cross-correlation ──────────────────────────────────
def conv2d_naive(
    x: np.ndarray,
    kernel: np.ndarray,
    stride: int = 1,
    padding: int = 0,
) -> np.ndarray:
    '''Compute 2-D cross-correlation using explicit nested loops.

    This slow implementation mirrors the mathematical definition exactly.
    Useful for understanding the operation, not for production use.

    Args:
        x: Input array of shape (H, W).
        kernel: Filter array of shape (kH, kW).
        stride: Step size for sliding the kernel.
        padding: Number of zero-padding rows/columns added on each side.

    Returns:
        Output feature map of shape (H_out, W_out).
    '''
    if padding > 0:
        x = np.pad(x, padding, mode="constant", constant_values=0.0)
    H, W   = x.shape
    kH, kW = kernel.shape
    H_out  = (H - kH) // stride + 1
    W_out  = (W - kW) // stride + 1
    out    = np.zeros((H_out, W_out), dtype=np.float32)
    for h in range(H_out):
        for w in range(W_out):
            patch      = x[h * stride: h * stride + kH, w * stride: w * stride + kW]
            out[h, w]  = float(np.sum(patch * kernel))
    return out


def output_size(in_size: int, kernel_size: int, stride: int = 1, padding: int = 0) -> int:
    '''Compute the output spatial size after a single convolution.

    Args:
        in_size: Input spatial dimension (height or width).
        kernel_size: Size of the square kernel.
        stride: Convolution stride.
        padding: Zero-padding on each side.

    Returns:
        Integer output spatial size.
    '''
    return (in_size + 2 * padding - kernel_size) // stride + 1


# ── 1-B: Vectorised (as_strided) — faster but harder to read ─────────────────
def conv2d_vectorized(
    x: np.ndarray,
    kernel: np.ndarray,
    stride: int = 1,
    padding: int = 0,
) -> np.ndarray:
    '''Vectorised 2-D cross-correlation using numpy stride tricks.

    Uses np.lib.stride_tricks.as_strided to build the sliding-window view
    without copying data, then collapses with a single einsum call.
    This is the numpy equivalent of the im2col strategy.

    Args:
        x: Input array of shape (H, W).
        kernel: Filter array of shape (kH, kW).
        stride: Kernel sliding step.
        padding: Zero-padding on each side.

    Returns:
        Output feature map of shape (H_out, W_out).
    '''
    if padding > 0:
        x = np.pad(x, padding, mode="constant", constant_values=0.0)
    H, W   = x.shape
    kH, kW = kernel.shape
    H_out  = (H - kH) // stride + 1
    W_out  = (W - kW) // stride + 1
    shape   = (H_out, W_out, kH, kW)
    strides = (x.strides[0] * stride, x.strides[1] * stride, x.strides[0], x.strides[1])
    patches = np.lib.stride_tricks.as_strided(x, shape=shape, strides=strides)
    return np.einsum("hwij,ij->hw", patches, kernel).astype(np.float32)


# ── Validate naive on a toy example ───────────────────────────────────────────
x_toy   = np.arange(1, 10, dtype=np.float32).reshape(3, 3)
k_toy   = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)  # Laplacian
out_toy = conv2d_naive(x_toy, k_toy, padding=1)
print("Toy input (3x3):\n", x_toy)
print("\nLaplacian kernel:\n", k_toy)
print("\nOutput (padding=1, stride=1):\n", out_toy)

# Output-size formula
print("\nOutput-size formula (28x28 input, 3x3 kernel):")
print(f"  {'pad':>3s}  {'stride':>6s}  {'H_out':>6s}")
for pad, str_ in [(0, 1), (1, 1), (0, 2), (1, 2)]:
    h_out = output_size(28, 3, stride=str_, padding=pad)
    print(f"  {pad:>3d}  {str_:>6d}  {h_out:>6d}")

# ── Correctness and speed comparison ──────────────────────────────────────────
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
out_naive = conv2d_naive(raw_sample_img, sobel_x, padding=1)
out_vec   = conv2d_vectorized(raw_sample_img, sobel_x, padding=1)
max_diff  = np.abs(out_naive - out_vec).max()
print(f"\nNaive vs vectorised max absolute error: {max_diff:.2e}  (should be ~0)")

REPS_TIMING = 10
t0 = time.perf_counter()
for _ in range(REPS_TIMING):
    conv2d_naive(raw_sample_img, sobel_x, padding=1)
t_naive = (time.perf_counter() - t0) / REPS_TIMING * 1000

t0 = time.perf_counter()
for _ in range(REPS_TIMING):
    conv2d_vectorized(raw_sample_img, sobel_x, padding=1)
t_vec = (time.perf_counter() - t0) / REPS_TIMING * 1000

print(f"\nTiming (28x28, 3x3 kernel, {REPS_TIMING} reps):")
print(f"  Naive (nested loops)    : {t_naive:.3f} ms")
print(f"  Vectorised (as_strided) : {t_vec:.3f} ms")
print(f"  Speedup                 : {t_naive / max(t_vec, 1e-9):.1f}x")


### 1.2 Classic Hand-Crafted Kernels

Before any learning, hand-engineered kernels reveal what kinds of spatial features
convolution can detect.  The same mathematical operation that picks out vertical edges
with a Sobel kernel will — after gradient descent — pick out curves, textures, and
object-level patterns.


In [ ]:
# ── Classic kernels dictionary ────────────────────────────────────────────────
KERNELS: dict[str, np.ndarray] = {
    "Identity":    np.array([[0,  0, 0], [ 0,  1,  0], [0,  0, 0]], dtype=np.float32),
    "Sobel X":     np.array([[-1, 0, 1], [-2,  0,  2], [-1, 0, 1]], dtype=np.float32),
    "Sobel Y":     np.array([[-1,-2,-1], [ 0,  0,  0], [ 1, 2, 1]], dtype=np.float32),
    "Laplacian":   np.array([[0,  1, 0], [ 1, -4,  1], [0,  1, 0]], dtype=np.float32),
    "Gauss Blur":  np.array([[1,  2, 1], [ 2,  4,  2], [1,  2, 1]], dtype=np.float32) / 16.0,
    "Sharpen":     np.array([[0, -1, 0], [-1,  5, -1], [0, -1, 0]], dtype=np.float32),
}

# Apply every kernel to a raw MNIST sample
n_k = len(KERNELS)
fig_k, axes_k = plt.subplots(2, n_k + 1, figsize=(2.8 * (n_k + 1), 5.5))

# Column 0: input image
axes_k[0][0].imshow(raw_sample_img, cmap="gray", vmin=0, vmax=1)
axes_k[0][0].set_title(f"Input\n(digit {raw_sample_lbl})", fontsize=9)
axes_k[0][0].axis("off")
axes_k[1][0].axis("off")

for col_idx, (name, kern) in enumerate(KERNELS.items(), start=1):
    feat_map = conv2d_naive(raw_sample_img, kern, padding=1)
    # Row 0: kernel weights heatmap
    v_k = np.abs(kern).max() or 1.0
    axes_k[0][col_idx].imshow(kern, cmap="RdBu_r", vmin=-v_k, vmax=v_k)
    axes_k[0][col_idx].set_title(name, fontsize=8)
    axes_k[0][col_idx].axis("off")
    # Row 1: resulting feature map
    v_f = np.abs(feat_map).max() or 1.0
    axes_k[1][col_idx].imshow(feat_map, cmap="RdBu_r", vmin=-v_f, vmax=v_f)
    axes_k[1][col_idx].set_title(f"{feat_map.shape}", fontsize=8)
    axes_k[1][col_idx].axis("off")

plt.suptitle("Row 0: kernel weights (red=+, blue=-)   Row 1: resulting feature map",
             fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

# Kernel statistics
print(f"{'Kernel':<15s}  {'feat min':>10s}  {'feat max':>10s}  {'feat std':>10s}")
print("-" * 52)
for name, kern in KERNELS.items():
    feat = conv2d_naive(raw_sample_img, kern, padding=1)
    print(f"  {name:<13s}  {feat.min():>10.3f}  {feat.max():>10.3f}  {feat.std():>10.3f}")


### 1.3 Padding & Stride

**Padding** (`p`): rows/columns of zeros around the border.
- `p = 0` (no padding) → output shrinks by $K-1$ pixels each side.
- `p = (K-1)/2` (same/half padding) → output preserves the spatial size.

**Stride** (`s`): how far the kernel moves per step.
- `s = 1` → dense, maximally overlapping receptive fields.
- `s = 2` → output is approximately halved; a stride-2 conv can replace MaxPool.


In [ ]:
# ── Visualise padding & stride effects ───────────────────────────────────────
edge_kernel = KERNELS["Sobel X"]
configs: list[dict] = [
    {"label": "pad=0, stride=1  (shrinks)",      "pad": 0, "stride": 1},
    {"label": "pad=1, stride=1  (same size)",    "pad": 1, "stride": 1},
    {"label": "pad=0, stride=2  (halves)",       "pad": 0, "stride": 2},
    {"label": "pad=1, stride=2  (half, padded)", "pad": 1, "stride": 2},
]

fig_ps, axes_ps = plt.subplots(1, len(configs), figsize=(14, 4))
for ax, cfg in zip(axes_ps, configs):
    feat = conv2d_naive(raw_sample_img, edge_kernel,
                        stride=cfg["stride"], padding=cfg["pad"])
    v = np.abs(feat).max() or 1.0
    ax.imshow(feat, cmap="RdBu_r", vmin=-v, vmax=v)
    ax.set_title(f"{cfg['label']}\nOutput: {feat.shape[0]}x{feat.shape[1]}", fontsize=8)
    ax.axis("off")

plt.suptitle("Sobel-X — Effect of Padding & Stride on Feature Map Size",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary table
print(f"  {'Config':<34s}  {'H_out':>6s}  {'W_out':>6s}  {'% of input':>10s}")
print("-" * 64)
for cfg in configs:
    h = output_size(28, 3, stride=cfg["stride"], padding=cfg["pad"])
    pct = h * h / (28 * 28) * 100
    print(f"  {cfg['label']:<34s}  {h:>6d}  {h:>6d}  {pct:>9.1f}%")

print("\nGeneral formula: H_out = floor((H + 2*p - K) / s) + 1")
print("Same-padding condition: p = (K - 1) / 2  =>  K=3 => p=1,  K=5 => p=2")


## Part 2 — Multi-Channel Convolution & `Conv2dScratch`

### 2.1 Multi-Channel Input and Multiple Filters

Real images have $C_{\text{in}}$ input channels (3 for RGB; 1 for grey-scale).
After the first layer, feature maps stack into $C_{\text{out}}$ channels.
The full cross-correlation extends to:

$$
Z_{c_{\text{out}},h,w} =
\sum_{c_{\text{in}}=0}^{C_{\text{in}}-1}\sum_{i=0}^{K_H-1}\sum_{j=0}^{K_W-1}
X_{c_{\text{in}},\,h \cdot s + i,\; w \cdot s + j}
\cdot K_{c_{\text{out}}, c_{\text{in}}, i, j} + b_{c_{\text{out}}}
$$

A weight tensor of shape $(C_{\text{out}}, C_{\text{in}}, K_H, K_W)$ has
$C_{\text{out}} \times C_{\text{in}} \times K_H \times K_W$ learnable parameters —
**shared** across all spatial positions $(h, w)$.

### 2.2 The im2col Strategy

Reshape the sliding-window patches of $\mathbf{X}$ into columns of a matrix, then reduce
the 3-nested-loop sum to a single **matrix multiply**:

$$
\mathbf{Z}_{\text{flat}} = \mathbf{W}_{\text{flat}} \cdot \mathbf{X}_{\text{col}}
$$

$\mathbf{W}_{\text{flat}} \in \mathbb{R}^{C_{\text{out}} \times C_{\text{in}} K_H K_W}$
and
$\mathbf{X}_{\text{col}} \in \mathbb{R}^{C_{\text{in}} K_H K_W \times H_{\text{out}} W_{\text{out}}}$.

PyTorch's `F.unfold` implements the column extraction; we then matmul and reshape.


In [ ]:
# ── im2col multi-channel cross-correlation ────────────────────────────────────
def conv2d_multichannel(
    x: torch.Tensor,
    weight: torch.Tensor,
    bias: torch.Tensor | None = None,
    stride: int = 1,
    padding: int = 0,
) -> torch.Tensor:
    '''Multi-channel 2-D cross-correlation via im2col (F.unfold + matmul).

    F.unfold extracts sliding-window columns, weight is flattened to a 2-D
    matrix, and a batched matmul replaces the three nested loops.

    Args:
        x: Input tensor of shape (N, C_in, H, W).
        weight: Filter tensor of shape (C_out, C_in, kH, kW).
        bias: Optional bias of shape (C_out,).
        stride: Kernel sliding step.
        padding: Zero-padding on each spatial side.

    Returns:
        Output feature maps of shape (N, C_out, H_out, W_out).
    '''
    N, C_in, H, W    = x.shape
    C_out, _, kH, kW = weight.shape
    H_out = (H + 2 * padding - kH) // stride + 1
    W_out = (W + 2 * padding - kW) // stride + 1

    # Step 1: im2col -> (N, C_in*kH*kW, H_out*W_out)
    x_col = F.unfold(x, kernel_size=(kH, kW), stride=stride, padding=padding)

    # Step 2: flatten weight -> (C_out, C_in*kH*kW)
    w_col = weight.view(C_out, -1)

    # Step 3: batched matmul -> (N, C_out, H_out*W_out)
    out = torch.bmm(w_col.unsqueeze(0).expand(N, -1, -1), x_col)

    if bias is not None:
        out = out + bias.view(1, -1, 1)

    return out.view(N, C_out, H_out, W_out)


# ── Correctness vs F.conv2d ───────────────────────────────────────────────────
torch.manual_seed(SEED)
x_test = torch.randn(2, 3, 8, 8)     # N=2, C_in=3, 8x8
w_test = torch.randn(8, 3, 3, 3)     # 8 filters, C_in=3, 3x3 kernel
b_test = torch.randn(8)

out_scratch = conv2d_multichannel(x_test, w_test, b_test, stride=1, padding=1)
out_ref     = F.conv2d(x_test, w_test, b_test, stride=1, padding=1)
max_err     = (out_scratch - out_ref).abs().max().item()
print(f"conv2d_multichannel vs F.conv2d: max absolute error = {max_err:.2e}")
assert max_err < 1e-5, "Implementation mismatch!"
print(f"Output shape: {out_scratch.shape}  (N=2, C_out=8, 8x8)  [OK]")

# ── Configuration sweep: parameters & output size ─────────────────────────────
print("\nConfiguration sweep (28x28 MNIST input, N=4, C_in=1):")
print(f"  {'C_out':>6s}  {'Kernel':>7s}  {'Pad':>4s}  {'Stride':>6s}  "
      f"{'Output':>12s}  {'Params':>8s}")
for c_out, k, pad, str_ in [(8, 3, 1, 1), (16, 3, 1, 1), (16, 5, 2, 1), (32, 3, 0, 2)]:
    h_out  = output_size(28, k, stride=str_, padding=pad)
    params = c_out * 1 * k * k + c_out    # weight + bias
    print(f"  {c_out:>6d}  {k:>4d}x{k:<2d}  {pad:>4d}  {str_:>6d}  "
          f"{h_out:>5d}x{h_out:<5d}  {params:>8,d}")


### 2.2b im2col Visualised

The im2col operation can be hard to picture.  Using a tiny 4×4 input and 2×2 kernel, we show exactly which elements of the input end up in each column of $\mathbf{X}_{\text{col}}$ — making the matrix-multiply interpretation concrete.


In [ ]:
# ── Visualising the im2col transformation ────────────────────────────────────
# F.unfold extracts every (kH, kW) patch at every valid (h, w) position and
# lays them as COLUMNS of a matrix X_col of shape (C_in*kH*kW, H_out*W_out).
# The weight matrix W_flat of shape (C_out, C_in*kH*kW) then does one matmul
# to compute ALL output positions simultaneously.

# Small example for clarity: 1-channel, 4x4 input, 2x2 kernel, no padding
x_small   = torch.arange(1, 17, dtype=torch.float32).reshape(1, 1, 4, 4)
w_small   = torch.ones(1, 1, 2, 2)   # sum-pooling kernel
print("Input (1x1x4x4):")
print(x_small[0, 0].numpy())

# im2col via F.unfold: (N=1, C_in*kH*kW=4, H_out*W_out=9)
x_col_vis = F.unfold(x_small, kernel_size=2, stride=1, padding=0)
print(f"\nAfter F.unfold (kernel=2x2, stride=1, pad=0):")
print(f"  Shape: {x_col_vis.shape}  = (N=1, C_in*kH*kW=4, H_out*W_out=9)")
print("  X_col matrix (each column = one 2x2 patch flattened):")
print(x_col_vis[0].numpy())

# Matrix multiply to get output
w_flat_vis = w_small.view(1, -1)   # (1, 4) — sum all 4 patch elements
out_flat   = w_flat_vis @ x_col_vis[0]   # (1, 9)
out_vis    = out_flat.view(1, 3, 3)
print(f"\nW_flat (1x4): {w_flat_vis.numpy()}")
print(f"W_flat @ X_col -> output (1x3x3):")
print(out_vis[0].numpy())

# Verify against F.conv2d
ref_out = F.conv2d(x_small, w_small, stride=1, padding=0)
max_err = (out_vis - ref_out).abs().max().item()
print(f"\nMax error vs F.conv2d: {max_err:.2e}  (should be 0)")

# ── Visualise the patch layout ─────────────────────────────────────────────────
fig_imcol, axes_imcol = plt.subplots(1, 2, figsize=(12, 5))

# Left: input with patch overlay
ax_in = axes_imcol[0]
inp_np = x_small[0, 0].numpy()
ax_in.imshow(inp_np, cmap="Blues", vmin=0, vmax=16)
for val, (ri, ci) in enumerate(np.ndindex(inp_np.shape)):
    ax_in.text(ci, ri, f"{inp_np[ri, ci]:.0f}", ha="center", va="center", fontsize=12)
# Highlight the first patch (top-left 2x2)
rect = mpatches.FancyBboxPatch((-0.5, -0.5), 2, 2,
                                boxstyle="round,pad=0.1",
                                edgecolor="red", facecolor="none", linewidth=2)
ax_in.add_patch(rect)
ax_in.set_title("4x4 Input (red box = first 2x2 patch)", fontsize=10)
ax_in.set_xticks([]); ax_in.set_yticks([])

# Right: X_col columns (each = one patch)
ax_col = axes_imcol[1]
ax_col.imshow(x_col_vis[0].numpy(), cmap="Blues", aspect="auto")
for ri in range(x_col_vis.shape[1]):
    for ci in range(x_col_vis.shape[2]):
        ax_col.text(ci, ri, f"{x_col_vis[0, ri, ci]:.0f}",
                    ha="center", va="center", fontsize=9)
ax_col.set_xlabel("Position index (H_out * W_out = 9 positions)", fontsize=9)
ax_col.set_ylabel("Patch elements (kH * kW = 4)", fontsize=9)
ax_col.set_title("X_col after F.unfold  (cols = flattened patches)", fontsize=10)

plt.suptitle("im2col: Turning the Sliding Window into a Matrix Multiply",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


### 2.3 `Conv2dScratch` — A Full `nn.Module`

We wrap the im2col approach in an `nn.Module` so it integrates with PyTorch's autograd,
optimisers, and `nn.Sequential`.  Weight initialisation mirrors `nn.Conv2d` defaults:
Kaiming uniform for filters, uniform from $[-1/\sqrt{\text{fan\_in}},\, +1/\sqrt{\text{fan\_in}}]$
for bias.


In [ ]:
class Conv2dScratch(nn.Module):
    '''2-D convolutional layer implemented from scratch via im2col.

    Uses F.unfold + batched matmul, matching the im2col strategy used in
    optimised BLAS-backed libraries.  Initialisation mirrors nn.Conv2d defaults.

    Attributes:
        in_channels: Number of input feature-map channels.
        out_channels: Number of output feature-map channels (filters).
        kernel_size: Height = width of each square kernel.
        stride: Step size for sliding the kernel.
        padding: Zero-padding added on each spatial side.
        weight: Learnable filter weights (out_channels, in_channels, kH, kW).
        bias: Learnable bias (out_channels,), or None when bias=False.
    '''

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        stride: int = 1,
        padding: int = 0,
        bias: bool = True,
    ) -> None:
        '''Initialise Conv2dScratch with Kaiming-uniform weight and bounded bias.

        Args:
            in_channels: Number of input feature-map channels.
            out_channels: Number of output feature-map channels.
            kernel_size: Square kernel height and width.
            stride: Kernel sliding step.
            padding: Zero-padding on each spatial side.
            bias: Whether to include a learnable bias term.
        '''
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.kernel_size  = kernel_size
        self.stride       = stride
        self.padding      = padding

        # Kaiming uniform — same default as nn.Conv2d
        self.weight = nn.Parameter(
            torch.empty(out_channels, in_channels, kernel_size, kernel_size)
        )
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

        if bias:
            fan_in = in_channels * kernel_size * kernel_size
            bound  = 1.0 / math.sqrt(fan_in) if fan_in > 0 else 0.0
            self.bias = nn.Parameter(
                torch.empty(out_channels).uniform_(-bound, bound)
            )
        else:
            self.register_parameter("bias", None)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Apply 2-D cross-correlation via im2col.

        Args:
            x: Input tensor of shape (N, C_in, H, W).

        Returns:
            Output feature maps of shape (N, C_out, H_out, W_out).
        '''
        N, _, H, W = x.shape
        kH = kW    = self.kernel_size
        H_out = (H + 2 * self.padding - kH) // self.stride + 1
        W_out = (W + 2 * self.padding - kW) // self.stride + 1

        x_col = F.unfold(x, kernel_size=kH, stride=self.stride, padding=self.padding)
        w_col = self.weight.view(self.out_channels, -1)
        out   = torch.bmm(w_col.unsqueeze(0).expand(N, -1, -1), x_col)

        if self.bias is not None:
            out = out + self.bias.view(1, -1, 1)

        return out.view(N, self.out_channels, H_out, W_out)

    def extra_repr(self) -> str:
        '''Return string summary of the layer configuration.'''
        return (f"in_channels={self.in_channels}, out_channels={self.out_channels}, "
                f"kernel_size={self.kernel_size}, stride={self.stride}, "
                f"padding={self.padding}")


# ── Verify against nn.Conv2d ──────────────────────────────────────────────────
torch.manual_seed(SEED + 1)
scratch_layer = Conv2dScratch(1, 16, kernel_size=3, stride=1, padding=1)
ref_layer     = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)

# Copy identical weights so outputs must match
ref_layer.weight.data.copy_(scratch_layer.weight.data)
ref_layer.bias.data.copy_(scratch_layer.bias.data)

x_verify = torch.randn(4, 1, 28, 28)
with torch.no_grad():
    out_s = scratch_layer(x_verify)
    out_r = ref_layer(x_verify)

max_err_mod = (out_s - out_r).abs().max().item()
print(f"Conv2dScratch vs nn.Conv2d: max absolute error = {max_err_mod:.2e}")
assert max_err_mod < 1e-5, "Module output mismatch!"
print(f"Output shape: {out_s.shape}  [OK]")
print(scratch_layer)

# Parameter count comparison
n_conv   = sum(p.numel() for p in scratch_layer.parameters())
n_dense  = 784 * 16    # dense layer equivalent for same # of output neurons
print(f"\nParameter count:")
print(f"  Conv2dScratch(1->16, k=3, p=1) : {n_conv:,}")
print(f"  Dense Linear (784->16)          : {n_dense:,}  ({n_dense // n_conv}x more)")


## Part 3 — Training a CNN on MNIST

### SimpleCNN Architecture

We build a compact three-layer CNN using `nn.Conv2d` (the verified library version) for
training efficiency.  The architecture demonstrates the spatial hierarchy CNNs construct:

| Layer | Operation | Channels | Spatial | Params |
|-------|-----------|----------|---------|--------|
| conv1 | Conv2d(k=3,p=1) + ReLU + MaxPool(2) | 1→16 | 28→14 | 160 |
| conv2 | Conv2d(k=3,p=1) + ReLU + MaxPool(2) | 16→32 | 14→7 | 4,640 |
| conv3 | Conv2d(k=3,p=1) + ReLU + AdaptAvgPool(4) | 32→64 | 7→4 | 18,496 |
| fc1   | Linear + ReLU + Dropout(0.3) | 1024→256 | — | 262,400 |
| fc2   | Linear | 256→10 | — | 2,570 |

Weight sharing means `conv1` uses only **160 parameters** to scan the full $28 \times 28$
input — an MLP's equivalent first layer would need $784 \times 16 = 12{,}544$.


In [ ]:
class SimpleCNN(nn.Module):
    '''Compact 3-layer CNN for MNIST digit classification.

    Architecture: three Conv+ReLU+Pool blocks feeding into a two-layer
    fully-connected head.

    Attributes:
        features: Sequential convolutional feature extractor.
        classifier: Sequential fully-connected classifier head.
    '''

    def __init__(self, num_classes: int = 10, dropout_rate: float = 0.3) -> None:
        '''Build SimpleCNN feature extractor and classifier head.

        Args:
            num_classes: Number of output classes.
            dropout_rate: Dropout probability in the classifier head.
        '''
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 28x28 -> 14x14
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 14x14 -> 7x7
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 7x7 -> 4x4
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Run forward pass through feature extractor and classifier.

        Args:
            x: Input images of shape (N, 1, 28, 28).

        Returns:
            Class logits of shape (N, num_classes).
        '''
        return self.classifier(self.features(x))


def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    '''Train the model for one epoch.

    Args:
        model: The CNN model.
        dataloader: Training DataLoader.
        criterion: Loss function.
        optimizer: Optimizer instance.
        device: Compute device.

    Returns:
        Tuple of (average_loss, accuracy) over the epoch.
    '''
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct      += logits.argmax(dim=1).eq(labels).sum().item()
        total        += labels.size(0)
    return running_loss / total, correct / total


def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    '''Evaluate the model on a data split.

    Args:
        model: The CNN model.
        dataloader: Validation or test DataLoader.
        criterion: Loss function.
        device: Compute device.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss   = criterion(logits, labels)
            running_loss += loss.item() * images.size(0)
            correct      += logits.argmax(dim=1).eq(labels).sum().item()
            total        += labels.size(0)
    return running_loss / total, correct / total


model     = SimpleCNN(num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

total_p   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"SimpleCNN total params: {total_p:,}  (trainable: {trainable:,})")
print(model)


In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
train_losses: list[float] = []
val_losses:   list[float] = []
train_accs:   list[float] = []
val_accs:     list[float] = []

best_val_loss = float("inf")
best_state    = None

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    elapsed = time.time() - t0

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.2%} | Time: {elapsed:.1f}s")

model.load_state_dict(best_state)
print(f"\nRestored best model (val loss = {best_val_loss:.4f})")


In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig_tr, axes_tr = plt.subplots(1, 2, figsize=(12, 4))
epochs_x = range(1, NUM_EPOCHS + 1)

axes_tr[0].plot(epochs_x, train_losses, marker="o", label="Train")
axes_tr[0].plot(epochs_x, val_losses,   marker="s", label="Val")
axes_tr[0].set_xlabel("Epoch"); axes_tr[0].set_ylabel("Loss")
axes_tr[0].set_title("Loss Curves"); axes_tr[0].legend(); axes_tr[0].grid(alpha=0.3)

axes_tr[1].plot(epochs_x, [a * 100 for a in train_accs], marker="o", label="Train")
axes_tr[1].plot(epochs_x, [a * 100 for a in val_accs],   marker="s", label="Val")
axes_tr[1].set_xlabel("Epoch"); axes_tr[1].set_ylabel("Accuracy (%)")
axes_tr[1].set_title("Accuracy Curves"); axes_tr[1].legend(); axes_tr[1].grid(alpha=0.3)

plt.suptitle("SimpleCNN Training on MNIST", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Final train acc : {train_accs[-1]:.2%}   val acc : {val_accs[-1]:.2%}")


### 3.2 Convolution Sweep: Kernel Size & Filter Count vs Runtime

Before training, we profile `nn.Conv2d` across different configurations to build intuition for how kernel size and filter count affect inference time and parameter count.  This motivates the VGG insight: two 3×3 convolutions cover the same receptive field as one 5×5 but use fewer parameters.


In [ ]:
# ── Convolution sweep: kernel size and filter count vs inference time ──────────
# Understanding how kernel size and filter count affect runtime guides
# architecture decisions — larger kernels have more parameters but also more
# FLOPs per position.

sweep_configs: list[dict] = [
    {"C_out": 8,  "k": 3},
    {"C_out": 16, "k": 3},
    {"C_out": 32, "k": 3},
    {"C_out": 64, "k": 3},
    {"C_out": 16, "k": 5},
    {"C_out": 16, "k": 7},
]

x_sweep      = torch.randn(32, 1, 28, 28)
SWEEP_REPS   = 30
WARMUP_REPS  = 5

sweep_results: list[dict] = []
for cfg in sweep_configs:
    c_out, k = cfg["C_out"], cfg["k"]
    pad      = k // 2    # same-padding
    layer    = nn.Conv2d(1, c_out, kernel_size=k, padding=pad)

    # Warm-up
    with torch.no_grad():
        for _ in range(WARMUP_REPS):
            layer(x_sweep)

    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(SWEEP_REPS):
            layer(x_sweep)
    elapsed_ms = (time.perf_counter() - t0) / SWEEP_REPS * 1000

    h_out    = output_size(28, k, stride=1, padding=pad)
    n_params = sum(p.numel() for p in layer.parameters())
    flops    = 2 * c_out * 1 * k * k * h_out * h_out * 32   # per batch

    sweep_results.append({
        "C_out":    c_out,
        "Kernel":   f"{k}x{k}",
        "Pad":      pad,
        "H_out":    h_out,
        "Params":   n_params,
        "FLOPs":    flops,
        "Time(ms)": elapsed_ms,
    })

sweep_df = pd.DataFrame(sweep_results)
print("Convolution sweep (C_in=1, N=32, input 28x28):")
print(sweep_df.to_string(index=False))

# Bar chart: latency per configuration
fig_sw, axes_sw = plt.subplots(1, 2, figsize=(12, 4))
labels = [f"C_out={r['C_out']}\n{r['Kernel']}" for r in sweep_results]
times  = [r["Time(ms)"] for r in sweep_results]
params = [r["Params"] for r in sweep_results]

axes_sw[0].bar(range(len(labels)), times, color="#1f77b4", edgecolor="k", lw=0.7)
axes_sw[0].set_xticks(range(len(labels)))
axes_sw[0].set_xticklabels(labels, fontsize=8)
axes_sw[0].set_ylabel("Inference time (ms/batch)", fontsize=10)
axes_sw[0].set_title("Latency by Conv Config (N=32)", fontsize=10, fontweight="bold")
axes_sw[0].grid(axis="y", alpha=0.3)

axes_sw[1].bar(range(len(labels)), params, color="#ff7f0e", edgecolor="k", lw=0.7)
axes_sw[1].set_xticks(range(len(labels)))
axes_sw[1].set_xticklabels(labels, fontsize=8)
axes_sw[1].set_ylabel("Number of parameters", fontsize=10)
axes_sw[1].set_title("Parameter Count by Conv Config", fontsize=10, fontweight="bold")
axes_sw[1].grid(axis="y", alpha=0.3)

plt.suptitle("Conv2d: Kernel Size and Filter Count vs Runtime & Parameters",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nKey observations:")
print("  - Doubling C_out roughly doubles FLOPs and latency.")
print("  - Larger kernels (5x5, 7x7) increase params quadratically but are often")
print("    replaced by stacked 3x3 convolutions in modern architectures (VGG insight).")


## Part 4 — Evaluation & Analysis

### 4.1 Test Set Evaluation

We report results on the held-out official 10 K MNIST test split — images never seen
during training or validation.


In [ ]:
# ── Test accuracy & confusion matrix ──────────────────────────────────────────
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}  |  Test Accuracy: {test_acc:.2%}")

all_preds:  list[int] = []
all_labels: list[int] = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.tolist())

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig_cm, ax_cm = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
disp.plot(ax=ax_cm, colorbar=False, cmap="Blues")
ax_cm.set_title(f"Confusion Matrix — SimpleCNN  (Test Acc: {test_acc:.2%})",
                fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# Per-class accuracy table
print("\nPer-class accuracy:")
print(f"  {'Digit':>5s}  {'Correct':>8s}  {'Total':>7s}  {'Acc':>7s}")
print("-" * 35)
for digit in range(10):
    mask      = [lab == digit for lab in all_labels]
    cls_preds = [p for p, m in zip(all_preds,  mask) if m]
    cls_lbls  = [l for l, m in zip(all_labels, mask) if m]
    n_correct = sum(p == l for p, l in zip(cls_preds, cls_lbls))
    cls_acc   = n_correct / max(len(cls_lbls), 1)
    print(f"  {digit:>5d}  {n_correct:>8d}  {len(cls_lbls):>7d}  {cls_acc:>6.2%}")


### 4.2 Feature Map Visualisation

We extract `conv1` activations for a sample digit, comparing **untrained** (random
He-init) vs **trained** feature maps.  This visualises what the first layer has learned:
before training the responses are noisy; after training they resemble edge/curve detectors.


In [ ]:
# ── Extract conv1 activations ─────────────────────────────────────────────────
def get_conv1_feature_maps(
    cnn_model: nn.Module,
    image: torch.Tensor,
    cnn_device: torch.device,
) -> tuple[torch.Tensor, torch.Tensor]:
    '''Extract first conv layer outputs (pre- and post-ReLU) for one image.

    Args:
        cnn_model: A SimpleCNN instance.
        image: Single image tensor of shape (1, 1, 28, 28).
        cnn_device: Compute device for the model.

    Returns:
        Tuple of (pre_relu, post_relu) tensors, each (1, 16, 28, 28).
    '''
    conv1 = cnn_model.features[0]   # nn.Conv2d(1, 16, ...)
    relu1 = cnn_model.features[1]   # nn.ReLU
    x_dev = image.to(cnn_device)
    with torch.no_grad():
        raw  = conv1(x_dev)
        relu = relu1(raw)
    return raw.cpu(), relu.cpu()


sample_img_tensor = test_set[0][0].unsqueeze(0)   # (1, 1, 28, 28)
sample_digit_lbl  = int(test_set[0][1])

untrained_cnn = SimpleCNN(num_classes=NUM_CLASSES)
torch.manual_seed(SEED + 42)
_, maps_untrained = get_conv1_feature_maps(untrained_cnn, sample_img_tensor, torch.device("cpu"))
_, maps_trained   = get_conv1_feature_maps(model, sample_img_tensor, device)

# 4-row grid: rows 0-1 = untrained, rows 2-3 = trained
fig_fm, axes_fm = plt.subplots(4, 8, figsize=(16, 9))
fig_fm.suptitle(
    f"conv1 Feature Maps (after ReLU) — Digit '{sample_digit_lbl}'\n"
    "Rows 0-1: Untrained (random He init)   Rows 2-3: Trained",
    fontsize=10, fontweight="bold",
)
for k in range(16):
    col    = k % 8
    ax_row = k // 8
    for row_base, maps in [(0, maps_untrained), (2, maps_trained)]:
        feat = maps[0, k].numpy()
        v    = np.abs(feat).max() or 1.0
        axes_fm[row_base + ax_row][col].imshow(feat, cmap="RdBu_r", vmin=-v, vmax=v)
        axes_fm[row_base + ax_row][col].set_title(f"f{k}", fontsize=7)
        axes_fm[row_base + ax_row][col].axis("off")
plt.tight_layout()
plt.show()

# Trained kernel heatmaps
conv1_w = model.features[0].weight.detach().cpu().numpy()  # (16, 1, 3, 3)
fig_kv, axes_kv = plt.subplots(2, 8, figsize=(16, 4))
fig_kv.suptitle("Trained conv1 Kernel Weights — 16 filters (1x3x3 each)",
                fontsize=10, fontweight="bold")
for k in range(16):
    ax  = axes_kv[k // 8][k % 8]
    ker = conv1_w[k, 0]
    v   = np.abs(ker).max() or 1.0
    ax.imshow(ker, cmap="RdBu_r", vmin=-v, vmax=v)
    ax.set_title(f"K{k}", fontsize=7)
    ax.axis("off")
plt.tight_layout()
plt.show()

# Activation statistics: trained vs untrained
print("Activation stats after conv1 + ReLU  (16 filters, 28x28):")
print(f"  {'Source':<12s}  {'mean':>8s}  {'std':>8s}  {'sparsity (ReLU zeros)':>22s}")
for label, maps in [("Untrained", maps_untrained), ("Trained", maps_trained)]:
    arr      = maps[0].numpy()
    sparsity = (arr == 0).mean()
    print(f"  {label:<12s}  {arr.mean():>8.4f}  {arr.std():>8.4f}  {sparsity:>21.2%}")


### 4.3 Error Analysis

Inspecting misclassified examples reveals which digit pairs confuse the network most.
Common failure patterns often involve structurally similar digits (4 vs 9, 3 vs 8).


In [ ]:
# ── Collect misclassified examples ────────────────────────────────────────────
errors: list[dict] = []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        preds  = logits.argmax(dim=1).cpu()
        for img, true_lbl, pred_lbl in zip(images, labels, preds):
            if true_lbl.item() != pred_lbl.item():
                errors.append({
                    "image":      img.squeeze().numpy(),
                    "true_label": true_lbl.item(),
                    "pred_label": pred_lbl.item(),
                })
        if len(errors) >= 40:
            break

print(f"Collected {len(errors)} misclassified examples")

# Visualise grid
n_show      = min(40, len(errors))
n_cols      = 10
n_rows      = math.ceil(n_show / n_cols)
fig_err, axes_err = plt.subplots(n_rows, n_cols, figsize=(15, 1.8 * n_rows))
axes_err_flat = axes_err.ravel() if n_rows > 1 else list(axes_err)

for idx in range(n_show):
    err      = errors[idx]
    ax       = axes_err_flat[idx]
    disp_img = err["image"] * MNIST_STD[0] + MNIST_MEAN[0]   # invert normalisation
    ax.imshow(disp_img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"T:{err['true_label']} P:{err['pred_label']}", fontsize=7, color="red")
    ax.axis("off")
for idx in range(n_show, len(axes_err_flat)):
    axes_err_flat[idx].axis("off")

plt.suptitle("Misclassified Test Digits (T=true label, P=predicted label)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# Most confused pairs
pair_counts: dict[tuple[int, int], int] = {}
for err in errors:
    key = (err["true_label"], err["pred_label"])
    pair_counts[key] = pair_counts.get(key, 0) + 1

sorted_pairs = sorted(pair_counts.items(), key=lambda x: -x[1])
print("\nTop confusion pairs (true -> predicted):")
for (true_lbl, pred_lbl), count in sorted_pairs[:8]:
    print(f"  Digit {true_lbl} mistaken for {pred_lbl} : {count} times")


### 4.4 Speed & Parameter Comparison

We benchmark `Conv2dScratch` (im2col) against `nn.Conv2d` and summarise the
parameter-efficiency advantage of convolution vs fully-connected layers.


In [ ]:
# ── Timing: Conv2dScratch vs nn.Conv2d ────────────────────────────────────────
batch_bm  = torch.randn(32, 1, 28, 28)
conv_ref  = nn.Conv2d(1, 16, 3, padding=1)
conv_scr  = Conv2dScratch(1, 16, 3, padding=1)
conv_scr.weight.data.copy_(conv_ref.weight.data)
conv_scr.bias.data.copy_(conv_ref.bias.data)

N_REPS = 50
with torch.no_grad():        # warm-up
    for _ in range(5):
        conv_ref(batch_bm); conv_scr(batch_bm)

t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(N_REPS):
        conv_ref(batch_bm)
t_ref = (time.perf_counter() - t0) / N_REPS * 1000

t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(N_REPS):
        conv_scr(batch_bm)
t_scr = (time.perf_counter() - t0) / N_REPS * 1000

with torch.no_grad():
    timing_err = (conv_ref(batch_bm) - conv_scr(batch_bm)).abs().max().item()

# ── Summary table ─────────────────────────────────────────────────────────────
rows = [
    {
        "Approach":     "nn.Conv2d (library)",
        "Latency (ms)": f"{t_ref:.3f}",
        "Max error":    "-- (reference)",
        "Params":       f"{sum(p.numel() for p in conv_ref.parameters()):,}",
    },
    {
        "Approach":     "Conv2dScratch (im2col)",
        "Latency (ms)": f"{t_scr:.3f}",
        "Max error":    f"{timing_err:.2e}",
        "Params":       f"{sum(p.numel() for p in conv_scr.parameters()):,}",
    },
    {
        "Approach":     "Dense Linear (784->16) [hypothetical]",
        "Latency (ms)": "--",
        "Max error":    "--",
        "Params":       f"{784 * 16:,}",
    },
]
cmp_df = pd.DataFrame(rows)
print("Comparison — correctness, speed, and parameter count:")
print(cmp_df.to_string(index=False))

conv_params  = sum(p.numel() for p in conv_ref.parameters())
dense_params = 784 * 16
print(f"\nParameter reduction (conv vs dense) : {dense_params / conv_params:.0f}x fewer")
print(f"Speed overhead (scratch / library)   : {t_scr / max(t_ref, 1e-9):.2f}x")
print(f"\nFinal SimpleCNN test accuracy        : {test_acc:.2%}")
print(f"SimpleCNN total parameters           : {sum(p.numel() for p in model.parameters()):,}")


# ── FLOPs-per-second throughput ───────────────────────────────────────────────
# For a real deployment metric, FLOPs/sec (throughput) combines latency and
# model complexity.  Higher FLOPs/sec means the hardware is better utilised.
BATCH_BENCH = 32
x_bench_full = torch.randn(BATCH_BENCH, 1, 28, 28)
model_bench  = model.cpu().eval()

BENCH_REPS = 20
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(BENCH_REPS):
        model_bench(x_bench_full)
t_full_ms = (time.perf_counter() - t0) / BENCH_REPS * 1000

# Approximate total FLOPs per image (2 * MAC operations)
flops_per_img = (
    2 * 16 * 1 * 3 * 3 * 28 * 28 +    # conv1
    2 * 32 * 16 * 3 * 3 * 14 * 14 +   # conv2
    2 * 64 * 32 * 3 * 3 * 7 * 7 +     # conv3
    2 * (64 * 4 * 4) * 256 +           # fc1
    2 * 256 * 10                        # fc2
)
total_flops = flops_per_img * BATCH_BENCH
gflops_per_sec = (total_flops / 1e9) / (t_full_ms / 1000)

model.to(device)   # restore device
print(f"\nSimpleCNN throughput ({BATCH_BENCH} images):")
print(f"  Latency   : {t_full_ms:.2f} ms/batch")
print(f"  Throughput: {BATCH_BENCH / (t_full_ms / 1000):.0f} images/sec")
print(f"  GFLOPs/s  : {gflops_per_sec:.3f}")
print(f"  FLOPs/img : {flops_per_img:,}  ({flops_per_img / 1e6:.3f} MFLOPs)")


### 4.5 Layer-by-Layer Analysis: Shapes, Parameters & FLOPs

A layer-by-layer trace shows how spatial dimensions shrink through the network and where parameters/compute are concentrated.  For image models, the convolutional layers use surprisingly few parameters while the fully-connected head dominates the count.


In [ ]:
# ── Layer-by-layer analysis: shapes, parameters, FLOPs ───────────────────────
# For each layer in SimpleCNN we track the output spatial size,
# parameter count, and approximate FLOPs (multiply-adds) for a single
# forward pass on one image.

def analyze_simplecnn(
    cnn_model: nn.Module,
    input_shape: tuple[int, int, int, int],
    cnn_device: torch.device,
) -> list[dict]:
    '''Trace SimpleCNN layer by layer, recording shape and parameter stats.

    Args:
        cnn_model: A SimpleCNN instance.
        input_shape: Batch input shape (N, C, H, W).
        cnn_device: Device for the model.

    Returns:
        List of dicts with layer name, output shape, params, and FLOPs.
    '''
    rows: list[dict] = []
    x = torch.zeros(input_shape).to(cnn_device)
    cnn_model.eval()

    def fmt_shape(t: torch.Tensor) -> str:
        '''Return spatial shape as a string, dropping the batch dimension.

        Args:
            t: Tensor whose shape we want to format.

        Returns:
            String like "16x28x28" (channels x H x W).
        '''
        return "x".join(str(d) for d in t.shape[1:])   # drop batch dim

    with torch.no_grad():
        for name, layer in cnn_model.named_modules():
            if isinstance(layer, (nn.Conv2d, nn.Linear, nn.MaxPool2d,
                                   nn.AdaptiveAvgPool2d, nn.ReLU,
                                   nn.Dropout, nn.Flatten)):
                x_in  = x.clone()
                x     = layer(x)
                n_p   = sum(p.numel() for p in layer.parameters())

                # Approximate FLOPs (for one image, not full batch)
                if isinstance(layer, nn.Conv2d):
                    # 2 * C_out * C_in * kH * kW * H_out * W_out
                    _, c_out, h_out, w_out = x.shape
                    _, c_in,  h_in,  w_in  = x_in.shape
                    kH, kW   = layer.kernel_size if hasattr(layer.kernel_size, '__len__') else (layer.kernel_size, layer.kernel_size)
                    flops = 2 * c_out * c_in * kH * kW * h_out * w_out
                elif isinstance(layer, nn.Linear):
                    in_f, out_f = layer.in_features, layer.out_features
                    flops = 2 * in_f * out_f
                else:
                    flops = 0

                rows.append({
                    "Layer":       name or type(layer).__name__,
                    "Type":        type(layer).__name__,
                    "Output shape": fmt_shape(x),
                    "Params":      n_p,
                    "FLOPs":       flops,
                })

    return rows


analysis = analyze_simplecnn(model, (1, 1, 28, 28), device)
analysis_df = pd.DataFrame(analysis)

print("SimpleCNN layer-by-layer analysis (single image):")
print(analysis_df[["Layer", "Type", "Output shape", "Params", "FLOPs"]].to_string(index=False))

total_params_net = sum(r["Params"] for r in analysis)
total_flops_net  = sum(r["FLOPs"]  for r in analysis)
print(f"\nTotal trainable params : {total_params_net:,}")
print(f"Total FLOPs (1 image)  : {total_flops_net:,}  ({total_flops_net / 1e6:.3f} MFLOPs)")

# Pie chart: parameter share by layer type
type_params: dict[str, int] = {}
for row in analysis:
    if row["Params"] > 0:
        key = row["Type"]
        type_params[key] = type_params.get(key, 0) + row["Params"]

if type_params:
    fig_pie, ax_pie = plt.subplots(figsize=(7, 5))
    wedge_labels = [f"{k}\n({v:,})" for k, v in type_params.items()]
    ax_pie.pie(type_params.values(), labels=wedge_labels, autopct="%1.1f%%",
               startangle=140, textprops={"fontsize": 9})
    ax_pie.set_title("SimpleCNN Parameter Share by Layer Type",
                      fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

# Conv layers: parameter efficiency vs dense equivalent
print("\nConv vs Dense parameter comparison (per layer):")
print(f"  {'Layer':<20s}  {'Conv params':>12s}  {'Dense equiv':>12s}  {'Savings':>8s}")
print("-" * 60)
for row in analysis:
    if row["Type"] == "Conv2d":
        shape_str = row["Output shape"]   # e.g. "16x28x28"
        parts  = shape_str.split("x")
        c_out  = int(parts[0])
        h_out  = int(parts[1])
        # Input shape from params: weight is (C_out, C_in, kH, kW) + bias C_out
        # conv_params = C_out * C_in * kH * kW + C_out
        conv_p = row["Params"]
        # Dense equivalent: flatten input to (C_in * H_in * W_in) -> C_out neurons
        # Approximate: use the fact that we know the conv params
        print(f"  {row['Layer']:<20s}  {conv_p:>12,d}  (spatial sharing keeps this small)")


## Part 5 — Summary & Lessons Learned

### Key Takeaways

- **Cross-correlation, not convolution** — ML frameworks implement cross-correlation
  (no kernel flip). The distinction vanishes for learned weights, so the convention
  is universal across the field.
- **Weight sharing is the key advantage** — a $3 \times 3$ kernel uses only 9 shared
  weights to process the entire input. Compared to a dense layer, that is roughly
  $784 / 9 \approx 87\times$ fewer parameters per filter.
- **im2col turns convolution into GEMM** — `F.unfold` reshapes input patches into
  columns so multi-channel cross-correlation reduces to a batched matrix multiply.
  This is exactly how cuDNN accelerates convolution on GPU.
- **Padding preserves resolution; stride down-samples** — both are architectural
  choices with predictable effects via $\lfloor(H + 2p - K)/s\rfloor + 1$.
- **Trained kernels become feature detectors** — random init gives noisy feature maps;
  after 5 epochs on MNIST, `conv1` learns edge/curve detectors similar to classical
  hand-crafted Sobel and Laplacian filters.

### What's Next

→ **06-02 (Pooling & Receptive Fields)** introduces MaxPool, AvgPool, and global average
pooling, and derives the receptive-field formula showing how each neuron's "field of view"
grows through stacked conv+pool layers.
